![image.png](https://static.poder360.com.br/uploads/2024/08/NFL-Logo-848x477.jpg)

# <span style="color: #0d69c5">Tratamento de Dados da NFL de 2009 a 2017</span>

### <span style="color: #8bc0f5">Sobre o database</span>

O dataset **NFL Play by Play 2009-2017 (v4)**, disponível no Kaggle, reúne o registro de todas as jogadas da NFL em nove temporadas. Cada linha representa uma jogada, totalizando cerca de **450 mil registros e 100 colunas**, que cobrem contexto do jogo (times, quarto, down, distância), jogadores envolvidos, resultados, penalidades e métricas de probabilidade de pontos e vitória (como EPA e Win Probability).

#### <span style="color: #8bc0f5">Objetivo da atividade</span>

Construir um pipeline completo de **qualidade e armazenamento**: diagnosticar a estrutura e os problemas do dataset, corrigir a tipificação, tratar valores ausentes e inconsistências com decisões justificadas, e exportar o resultado em formato Parquet. Ao final, compara-se o tamanho com o CSV original, valida-se que os dados foram preservados e resume-se as melhorias obtidas em **consistência, confiabilidade e eficiência de armazenamento.**

### <span style="color: #8bc0f5">Carregamento de bibliotecas e database</span>

Nessa etapa realizamos o carregamento do Banco de Dados. Também realizamos a importação das bibliotecas necessárias para análise exploratória dos dados.

Bibliotecas Essenciais
- 🐼 Pandas
- 🔢 Numpy
- 📖 KaggleHub
- 💻 OS

In [2]:
#%pip install kagglehub

In [3]:
# Manipulação dos dados
import pandas as pd # Dataframe
import numpy as np # Operações
import os
import kagglehub
from kagglehub import KaggleDatasetAdapter

c:\Users\48694260808\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Download doataset do KaggleHub
root = kagglehub.dataset_download("maxhorowitz/nflplaybyplay2009to2016") 
print("Root:", root)

Root: C:\Users\48694260808\.cache\kagglehub\datasets\maxhorowitz\nflplaybyplay2009to2016\versions\6


In [5]:
# Criar caminho para o arquivo CSV 
csv_path = os.path.join(root, "NFL Play by Play 2009-2017 (v4).csv") 

# Carrega o arquivo CSV em um DataFrame
df = pd.read_csv(csv_path) 
df.head()

C:\Users\48694260808\AppData\Local\Temp\ipykernel_3348\1156612853.py:5: DtypeWarning: Columns (25: DefTwoPoint, 51: BlockingPlayer) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


,Date,GameID,Drive,qtr,down,time,TimeUnder,TimeSecs,PlayTimeDiff,SideofField,...,yacEPA,Home_WP_pre,Away_WP_pre,Home_WP_post,Away_WP_post,Win_Prob,WPA,airWPA,yacWPA,Season
0,2009-09-10,2009091000,1,1,NaN,15:00,15,3600.0,0.0,TEN,...,NaN,0.485675,0.514325,0.546433,0.453567,0.485675,0.060758,NaN,NaN,2009
1,2009-09-10,2009091000,1,1,1.0,14:53,15,3593.0,7.0,PIT,...,1.146076,0.546433,0.453567,0.551088,0.448912,0.546433,0.004655,-0.032244,0.036899,2009
2,2009-09-10,2009091000,1,1,2.0,14:16,15,3556.0,37.0,PIT,...,NaN,0.551088,0.448912,0.510793,0.489207,0.551088,-0.040295,NaN,NaN,2009
3,2009-09-10,2009091000,1,1,3.0,13:35,14,3515.0,41.0,PIT,...,-5.031425,0.510793,0.489207,0.461217,0.538783,0.510793,-0.049576,0.106663,-0.156239,2009
4,2009-09-10,2009091000,1,1,4.0,13:27,14,3507.0,8.0,PIT,...,NaN,0.461217,0.538783,0.558929,0.441071,0.461217,0.097712,NaN,NaN,2009


### <span style="color: #8bc0f5">Diagnóstico de Dados</span>

In [6]:
# Dimensões de Linhas X Colunas
df.shape

(407688, 102)

In [7]:
# Visualização de linhas aleatórias do DataFrame
df.sample(5)

,Date,GameID,Drive,qtr,down,time,TimeUnder,TimeSecs,PlayTimeDiff,SideofField,...,yacEPA,Home_WP_pre,Away_WP_pre,Home_WP_post,Away_WP_post,Win_Prob,WPA,airWPA,yacWPA,Season
349909,2016-12-04,2016120401,21,4,1.0,06:41,7,401.0,40.0,MIA,...,-1.379087,0.999782,0.000218,0.999720,0.000280,0.999782,-0.000062,0.000102,-0.000164,2016
128202,2011-12-18,2011121810,14,3,NaN,03:01,4,1081.0,0.0,CLE,...,NaN,0.165350,0.834650,0.233695,0.766305,0.165350,0.068345,NaN,NaN,2011
71738,2010-11-21,2010112110,4,1,2.0,04:07,5,2947.0,36.0,STL,...,NaN,0.600497,0.399503,0.635459,0.364541,0.399503,-0.034962,NaN,NaN,2010
166245,2012-12-02,2012120208,18,3,2.0,12:52,13,1672.0,3.0,HOU,...,NaN,0.054656,0.945344,0.049668,0.950332,0.945344,0.004988,NaN,NaN,2012
212265,2013-12-01,2013120104,8,2,2.0,10:54,11,2454.0,40.0,CHI,...,0.000000,0.370953,0.629047,0.376677,0.623323,0.629047,-0.005724,-0.005724,0.000000,2013


1. Date - Data
2. time - minute 
3. SideofField - category
4. posteam- category 
5. DefensiveTeam - category 
6. desc - string 
7. ExPointResult - category
8. TwoPointConv- boolean (failure e success)
9. DefTwoPoint - boolean (failure e success)
10. PuntResult - boolean (clean e blocked)
11. PlayType - category 
12. Passer - string
13. Passer_ID - string
14. PassOutcome - boolean (complete e incomplete pass)
15. PassLength - category
16. PassLocation - category
17. Interceptor - string
18. Rusher - string
19. Rusher_ID - string
20. RunLocation - category
21. RunGap - category
22. Receiver - string
23. Receiver_ID - string
24. ReturnResult - category
25. Returner - string 
26. BlockingPlayer - string
27. Tackler1 - string
28. Tackler2 - string
29. FieldGoalResult - category
30. RecFumbTeam - category
31. RecFumbPlayer - string
32. ChalReplayResult - category
33. PenalizedTeam - category
34. PenaltyType - category 
35. PenalizedPlayer - string
36. HomeTeam - category 
37. AwayTeam - category
38. Timeout_Team - category 


In [8]:
# Analisando Tipo de Dados das colunas 
df.dtypes

Date            str
GameID        int64
Drive         int64
qtr           int64
down        float64
             ...   
Win_Prob    float64
WPA         float64
airWPA      float64
yacWPA      float64
Season        int64
Length: 102, dtype: object

In [34]:
df.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 407688 entries, 0 to 407687
Data columns (total 102 columns):
 #    Column                       Dtype  
---   ------                       -----  
 0    Date                         str    
 1    GameID                       int64  
 2    Drive                        int64  
 3    qtr                          int64  
 4    down                         float64
 5    time                         str    
 6    TimeUnder                    int64  
 7    TimeSecs                     float64
 8    PlayTimeDiff                 float64
 9    SideofField                  str    
 10   yrdln                        float64
 11   yrdline100                   float64
 12   ydstogo                      int64  
 13   ydsnet                       int64  
 14   GoalToGo                     float64
 15   FirstDown                    float64
 16   posteam                      str    
 17   DefensiveTeam                str    
 18   desc                         str 

In [9]:
# Uso total da memória inicial em MB
memoria_inicial = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Uso total da memória inicial: {memoria_inicial:.2f} MB")

Uso total da memória inicial: 380.48 MB


In [12]:
# Uso de memória por coluna, em MB
memoria_por_coluna = (
    df.memory_usage(deep=True, index=False)
      .div(1024 ** 2)
      .sort_values(ascending=False)
)

display(memoria_por_coluna.rename("Memória (MB)").to_frame())

,Memória (MB)
desc,36.062976
Date,6.998428
time,5.097298
Tackler1,5.092860
PlayType,4.953390
...,...
Win_Prob,3.110413
WPA,3.110413
airWPA,3.110413
yacWPA,3.110413


In [17]:
# Compreendendo valores únicos e duplicados
resumo_colunas = pd.DataFrame({
    "Valores únicos": df.nunique(),
    "Valores duplicados": df.duplicated().sum()
})

display(resumo_colunas)

,Valores únicos,Valores duplicados
Date,426,0
GameID,2304,0
Drive,35,0
qtr,5,0
down,4,0
...,...,...
Win_Prob,360389,0
WPA,366549,0
airWPA,158705,0
yacWPA,135470,0


In [16]:
# Compreendendo valores únicos e sua contagem
for coluna in df.columns:
    valores_unicos = df[coluna].nunique()
    if valores_unicos < 30:
        print(f"Coluna: {coluna}")
        print(f"Valores únicos: {df[coluna].unique()}")
        print(f"Contagem e valores: {df[coluna].value_counts()}")
        print("*"*50)
print("FIM DA ANÁLISE")

Coluna: qtr
Valores únicos: [1 2 3 4 5]
Contagem e valores: qtr
4    112641
2    112317
3     90682
1     89176
5      2872
Name: count, dtype: int64
**************************************************
Coluna: down
Valores únicos: [nan  1.  2.  3.  4.]
Contagem e valores: down
1.0    138878
2.0    104089
3.0     67398
4.0     36169
Name: count, dtype: int64
**************************************************
Coluna: TimeUnder
Valores únicos: [15 14 13 12 11 10  9  8  7  6  5  4  3  2  1  0]
Contagem e valores: TimeUnder
2     40942
1     39254
15    32543
3     27305
4     26232
5     25072
7     23681
11    23661
6     23654
10    23582
8     23431
9     23397
13    23150
12    22981
14    21883
0      6920
Name: count, dtype: int64
**************************************************
Coluna: GoalToGo
Valores únicos: [ 0.  1. nan]
Contagem e valores: GoalToGo
0.0    386858
1.0     19990
Name: count, dtype: int64
**************************************************
Coluna: FirstDown
Valores 

In [19]:
# Verificando quantidade de nulos, percentual e valores válidos
resumo_nulos = pd.DataFrame({
    "Valores nulos": df.isna().sum(),
    "Valores válidos": df.notna().sum(),
    "Porcentagem nulos": round((df.isna().mean() * 100), 2)
})

display(resumo_nulos)

,Valores nulos,Valores válidos,Porcentagem nulos
Date,0,407688,0.00
GameID,0,407688,0.00
Drive,0,407688,0.00
qtr,0,407688,0.00
down,61154,346534,15.00
...,...,...,...
Win_Prob,25009,382679,6.13
WPA,5541,402147,1.36
airWPA,248501,159187,60.95
yacWPA,248762,158926,61.02


#### Porcentagem de Valores Nulos por Coluna

In [20]:
porcentagem_nulos = round((df.isna().mean() * 100), 2)

In [21]:
# Identificando colunas com percentual nulo acima de 50%
missing_50 = porcentagem_nulos[porcentagem_nulos >= 50].index
missing_50

Index(['ExPointResult', 'TwoPointConv', 'DefTwoPoint', 'PuntResult', 'Passer',
       'Passer_ID', 'PassOutcome', 'PassLength', 'PassLocation', 'Interceptor',
       'Rusher', 'Rusher_ID', 'RunLocation', 'RunGap', 'Receiver',
       'Receiver_ID', 'ReturnResult', 'Returner', 'BlockingPlayer', 'Tackler2',
       'FieldGoalResult', 'FieldGoalDistance', 'RecFumbTeam', 'RecFumbPlayer',
       'ChalReplayResult', 'PenalizedTeam', 'PenaltyType', 'PenalizedPlayer',
       'Timeout_Team', 'airEPA', 'yacEPA', 'airWPA', 'yacWPA'],
      dtype='str')

In [22]:
# Identificando colunas com percentual nulo entre 30 e 50%
missing_30_50 = porcentagem_nulos[porcentagem_nulos.between(30, 50, inclusive='right')].index
missing_30_50

Index(['Tackler1'], dtype='str')

In [23]:
# Identificando colunas com percentual nulo entre de 20 e 30%
missing_20_30 = porcentagem_nulos[porcentagem_nulos.between(20, 30, inclusive='right')].index
missing_20_30

Index([], dtype='str')

In [24]:
# Identificando colunas com percentual nulo entre de 10 e 20%
missing_10_20 = porcentagem_nulos[porcentagem_nulos.between(10, 20, inclusive='right')].index
missing_10_20

Index(['down'], dtype='str')

In [25]:
# Identificando colunas com percentual nulo entre de 5 e 10%
missing_5_10 = porcentagem_nulos[porcentagem_nulos.between(5, 10, inclusive='right')].index
missing_5_10

Index(['FirstDown', 'posteam', 'DefensiveTeam', 'PosTeamScore', 'DefTeamScore',
       'ScoreDiff', 'AbsScoreDiff', 'Home_WP_pre', 'Away_WP_pre',
       'Home_WP_post', 'Away_WP_post', 'Win_Prob'],
      dtype='str')

In [26]:
# Identificando colunas com percentual nulo entre de 0 e 5%
missing_5 = porcentagem_nulos[porcentagem_nulos.between(0, 5, inclusive='right')].index
missing_5

Index(['time', 'TimeSecs', 'PlayTimeDiff', 'SideofField', 'yrdln',
       'yrdline100', 'GoalToGo', 'No_Score_Prob', 'Opp_Field_Goal_Prob',
       'Opp_Safety_Prob', 'Opp_Touchdown_Prob', 'Field_Goal_Prob',
       'Safety_Prob', 'Touchdown_Prob', 'ExpPts', 'EPA', 'WPA'],
      dtype='str')

### <span style="color: #8bc0f5">Lista de possíveis inconsistências </span>


* Tipagem de dados (object/str -> tipo correto)
* DtypeWarning (colunas: DefTwoPoint e BlockingPlayer)
* Valores nulos

In [31]:
df_silver = df.copy()

In [53]:
# Verificar colunas que faltam para alterar a tipagem 
colunas_object = df_silver.select_dtypes(include=["str"]).columns.tolist()

print(f"Valores restantes para tratar tipagem: {len(colunas_object)}")
print("-"*30)
print("Colunas do tipo object:")
for col in colunas_object:
    print(f" - {col}")

Valores restantes para tratar tipagem: 14
------------------------------
Colunas do tipo object:
 - desc
 - Passer
 - Passer_ID
 - Interceptor
 - Rusher
 - Rusher_ID
 - Receiver
 - Receiver_ID
 - Returner
 - BlockingPlayer
 - Tackler1
 - Tackler2
 - RecFumbPlayer
 - PenalizedPlayer


In [45]:
for coluna in colunas_object:
    print(f"Coluna: {coluna}")
    print(f"Qnt Dados unicos {df[coluna].nunique()}")
    print(f"Qnt Dados unicos {df[coluna].unique()}")
    print("-"*50)
    print("\n")


Coluna: Date
Qnt Dados unicos 426
Qnt Dados unicos <ArrowStringArray>
['2009-09-10', '2009-09-13', '2009-09-14', '2009-09-20', '2009-09-21',
 '2009-09-27', '2009-09-28', '2009-10-04', '2009-10-05', '2009-10-11',
 ...
 '2017-12-10', '2017-12-11', '2017-12-14', '2017-12-16', '2017-12-17',
 '2017-12-18', '2017-12-23', '2017-12-24', '2017-12-25', '2017-12-31']
Length: 426, dtype: str
--------------------------------------------------


Coluna: time
Qnt Dados unicos 901
Qnt Dados unicos <ArrowStringArray>
['15:00', '14:53', '14:16', '13:35', '13:27', '13:16', '12:40', '12:11',
 '11:34', '11:24',
 ...
 '14:02', '09:58', '14:38', '09:46', '14:43', '14:58', '13:18', '14:36',
 '14:37', '14:34']
Length: 902, dtype: str
--------------------------------------------------


Coluna: SideofField
Qnt Dados unicos 37
Qnt Dados unicos <ArrowStringArray>
['TEN', 'PIT', 'MID', 'MIN', 'CLE', 'DET',  'NO',  'TB', 'DAL', 'NYJ', 'HOU',
 'JAC', 'IND', 'CIN', 'DEN',   nan, 'PHI', 'CAR', 'BAL',  'KC', 'MIA', 'AT

In [35]:
colunas_int64 = df_silver.select_dtypes(include=["int64"]).columns
for nome_coluna in colunas_int64:
    df_silver[nome_coluna] = pd.to_numeric(
        df_silver[nome_coluna],
        downcast="integer"
    )

print("Tipos resultantes:")
display(df_silver[colunas_int64].dtypes)

Tipos resultantes:


GameID                         int32
Drive                           int8
qtr                             int8
TimeUnder                       int8
ydstogo                         int8
ydsnet                          int8
PlayAttempted                   int8
Yards.Gained                    int8
sp                              int8
Touchdown                       int8
Safety                          int8
Onsidekick                      int8
PassAttempt                     int8
AirYards                        int8
YardsAfterCatch                 int8
QBHit                           int8
InterceptionThrown              int8
RushAttempt                     int8
Reception                       int8
Fumble                          int8
Sack                            int8
Challenge.Replay                int8
Accepted.Penalty                int8
Penalty.Yards                   int8
Timeout_Indicator               int8
posteam_timeouts_pre            int8
HomeTimeouts_Remaining_Pre      int8
A

In [37]:
colunas_float64 = df_silver.select_dtypes(include=["float64"]).columns

for nome_coluna in colunas_float64:
    df_silver[nome_coluna] = pd.to_numeric(
        df_silver[nome_coluna],
        downcast="float"
    )

print("Tipos resultantes:")
display(df_silver[colunas_float64].dtypes)

Tipos resultantes:


down                   float32
TimeSecs               float32
PlayTimeDiff           float32
yrdln                  float32
yrdline100             float32
GoalToGo               float32
FirstDown              float32
FieldGoalDistance      float32
PosTeamScore           float32
DefTeamScore           float32
ScoreDiff              float32
AbsScoreDiff           float32
No_Score_Prob          float32
Opp_Field_Goal_Prob    float32
Opp_Safety_Prob        float32
Opp_Touchdown_Prob     float32
Field_Goal_Prob        float32
Safety_Prob            float32
Touchdown_Prob         float32
ExPoint_Prob           float32
TwoPoint_Prob          float32
ExpPts                 float32
EPA                    float32
airEPA                 float32
yacEPA                 float32
Home_WP_pre            float32
Away_WP_pre            float32
Home_WP_post           float32
Away_WP_post           float32
Win_Prob               float32
WPA                    float32
airWPA                 float32
yacWPA  

In [46]:
colunas_category = [
    "SideofField",
    "posteam",
    "DefensiveTeam",
    "ExPointResult",
    "TwoPointConv",
    "DefTwoPoint",
    "PuntResult",
    "PlayType",
    "PassOutcome",
    "PassLength",
    "PassLocation",
    "RunLocation",
    "RunGap",
    "ReturnResult",
    "FieldGoalResult",
    "RecFumbTeam",
    "ChalReplayResult",
    "PenalizedTeam",
    "PenaltyType",
    "HomeTeam",
    "AwayTeam",
    "Timeout_Team",
]

df_silver[colunas_category] = (
    df_silver[colunas_category].astype("category")
)

print("Colunas alteradas de object para category")

Colunas alteradas de object para category


In [49]:
df_silver["Date"] = pd.to_datetime(df_silver["Date"], errors="coerce")

# Converte o relógio do jogo (MM:SS) em duração
df_silver["time"] = pd.to_timedelta(
    "00:" + df_silver["time"].astype("string"),
    errors="coerce"
)

print("Colunas Date e time convertidas")

Colunas Date e time convertidas


In [54]:
df_silver[colunas_object] = df_silver[colunas_object].astype("string[pyarrow]")
print("Colunas alteradas de object para String[pyarrow]")

Colunas alteradas de object para String[pyarrow]


In [55]:
memoria_antes_int = df[colunas_int64].memory_usage(deep=True).sum()
memoria_depois_int = df_silver[colunas_int64].memory_usage(deep=True).sum()

memoria_antes_float = df[colunas_float64].memory_usage(deep=True).sum()
memoria_depois_float = df_silver[colunas_float64].memory_usage(deep=True).sum()

mem_ant = df.memory_usage(deep=True).sum()
mem_dps = df_silver.memory_usage(deep=True).sum()

print(f"Memória antes:  {memoria_antes_int / (1024 ** 2):.2f} MB")
print(f"Memória depois: {memoria_depois_int / (1024 ** 2):.2f} MB")
print(f"Economia:       {(memoria_antes_int - memoria_depois_int) / (1024 ** 2):.2f} MB")
print("-"*50)
print(f"Memória antes:  {memoria_antes_float / (1024 ** 2):.2f} MB")
print(f"Memória depois: {memoria_depois_float / (1024 ** 2):.2f} MB")
print(f"Economia:       {(memoria_antes_float - memoria_depois_float) / (1024 ** 2):.2f} MB")
print("-"*50)
print(f"Memória antes:  {mem_ant / (1024 ** 2):.2f} MB")
print(f"Memória depois: {mem_dps / (1024 ** 2):.2f} MB")

Memória antes:  96.42 MB
Memória depois: 13.61 MB
Economia:       82.81 MB
--------------------------------------------------
Memória antes:  102.64 MB
Memória depois: 51.32 MB
Economia:       51.32 MB
--------------------------------------------------
Memória antes:  380.48 MB
Memória depois: 167.59 MB
